# 08 FastAPI Gateway

This notebook explains and starts the adapter routing gateway.

```mermaid
flowchart LR
    A[Client request] --> B[FastAPI gateway]
    B --> C{Adapter}
    C -->|finance| D[vLLM model finance]
    C -->|legal| E[vLLM model legal]
    C -->|healthcare| F[vLLM model healthcare]
    C -->|fallback| G[vLLM model base]
```

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Routing rules

The gateway accepts explicit `adapter` or `domain` fields. If none is provided, it uses simple domain keywords.

In [ ]:
from serving.gateway import ChatMessage, infer_adapter

examples = [
    "Explain revenue concentration risk.",
    "Summarize this limitation of liability clause.",
    "What does prior authorization mean?",
    "Tell me a neutral productivity tip.",
]
for prompt in examples:
    adapter = infer_adapter([ChatMessage(role="user", content=prompt)])
    print(f"{adapter}: {prompt}")

## Start gateway

Run this in a terminal. Expected output: Uvicorn on `http://localhost:8080`.

In [ ]:
print("make api")

## Test gateway request

Expected output: an OpenAI-compatible chat completion plus `routed_adapter`.

In [ ]:
import requests

payload = {
    "adapter": "finance",
    "messages": [{"role": "user", "content": "Explain revenue concentration risk."}],
    "temperature": 0.2,
    "max_tokens": 160,
}
response = requests.post(f"{settings_cfg.api_base_url}/chat", json=payload, timeout=60)
print(response.status_code)
print(response.text[:1200])